In [1]:
# LangSmith 추적 설정 부분
from dotenv import load_dotenv
import os

load_dotenv()

project_name = "wanted_2nd_retriever"
os.environ["LANGSMITH_PROJECT"] = project_name

In [37]:
# 출력 예쁘게 하기
from rich.console import Console
from rich.table import Table

console = Console()

def rich_docs(docs, max_len=140, title="Retriever Results"):
    table = Table(title=title)
    table.add_column("#", justify="right")
    table.add_column("Source")
    table.add_column("Page", justify="right")
    table.add_column("Preview")

    for i, d in enumerate(docs, 1):
        m = d.metadata or {}
        src = (m.get("source","") or "").split("/")[-1]
        page = str(m.get("page_label", m.get("page",0)+1))
        text = (d.page_content or "").strip().replace("\n", " ")
        table.add_row(str(i), src, page, (text[:max_len] + ("…" if len(text) > max_len else "")))
    console.print(table)


In [5]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

In [6]:
file_path = "../../data/Samsung_Electronics_Sustainability_Report_2025_KOR.pdf"

loader = PyPDFLoader(file_path)
docs = loader.load()
docs[:5]

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2025-07-10T16:11:16+09:00', 'moddate': '2025-09-04T16:51:11+09:00', 'trapped': '/False', 'source': '../../data/Samsung_Electronics_Sustainability_Report_2025_KOR.pdf', 'total_pages': 87, 'page': 0, 'page_label': '1'}, page_content='삼성전자 지속가능경영보고서 2025\nA Journey  Towards \n a Sustainable Future\nA Journey  Towards\n a Sustainable Future'),
 Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2025-07-10T16:11:16+09:00', 'moddate': '2025-09-04T16:51:11+09:00', 'trapped': '/False', 'source': '../../data/Samsung_Electronics_Sustainability_Report_2025_KOR.pdf', 'total_pages': 87, 'page': 1, 'page_label': '2'}, page_content='삼성전자 지속가능경영보고서 2025 02AppendixFacts & Figures PrinciplePlanet PeopleOur Company삼성전자 지속가능경영보고서 2025 02\nA Journey  Towards \n a Sustainable Future\nA Journey  Towards \n a Sustainable F

In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunk_docs = splitter.split_documents(docs)
len(chunk_docs)


237

In [8]:
# 벡터 저장소 만들기
embedding = OpenAIEmbeddings(model="text-embedding-3-small")
persist_directory = "../7_vectorstore/samsung_2025_db"
collection_name = "samsung2025"

In [ ]:
# vectorstore = Chroma.from_documents(
#     documents = chunk_docs,
#     embedding = embedding,
#     collection_name = collection_name,
#     persist_directory = persist_directory
# )

In [11]:
load_vectorstore = Chroma(
    collection_name=collection_name,
    persist_directory=persist_directory,
    embedding_function=embedding
)

load_vectorstore._collection.count()

237

### 1. 키워드기반 + 기본 검색기 = 하이브리드(ensemble)


In [12]:
# uv add rank-bm25

In [13]:
ret_similarity = load_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [15]:
# 번외: 메타 데이터가 궁금하다
db_docs = load_vectorstore._collection.get(include=["documents", "metadatas"])

In [17]:
db_docs["documents"][:5]

['삼성전자 지속가능경영보고서 2025\nA Journey  Towards \n a Sustainable Future\nA Journey  Towards\n a Sustainable Future',
 '삼성전자 지속가능경영보고서 2025 02AppendixFacts & Figures PrinciplePlanet PeopleOur Company삼성전자 지속가능경영보고서 2025 02\nA Journey  Towards \n a Sustainable Future\nA Journey  Towards \n a Sustainable Future\n삼성전자 지속가능경영보고서 2025\nCEO 메시지\n회사소개\n기업 지배구조\n중대성 평가\n이해관계자 소통\n준법과 윤리경영\n[DX부문] \n추진체계와 주요성과 \n기후변화 \n자원순환\n수자원\n오염물질\n경제성과\n사회성과\n환경성과 \n사업부문별 환경성과 \n[DS부문] \n추진체계와 주요성과 \n기후변화 \n자원순환\n수자원\n오염물질\n독립된 인증인의 인증보고서\nScope 1, 2 온실가스 배출량 검증 의견서 \nScope 3 온실가스 배출량 검증 의견서 \nGRI Index\nTCFD 대조표\nSASB 대조표\nAbout This Report \n임직원\n공급망\n사회공헌\n개인정보보호와 보안\n제품 품질과 안전\nOur Company\nPrinciple\nPlanet\nFacts & Figures Appendix \nPeople\n04\n05\n06\n07\n09\n59\n11\n12\n16\n18\n20\n62\n63\n68\n72\n21\n22\n27\n29\n32\n76\n77\n78\n80\n82\n84\n86\n35\n45\n51\n53\n55',
 '삼성전자 지속가능경영보고서 2025\n03\nOur Company AppendixFacts & Figures PrinciplePlanet People\nOur Company\nCEO 메시지\n회사소개\n기업 지배구조\n중대성 평가\n이해관계자 소통\n

In [18]:
db_docs["metadatas"][:5]

[{'creator': 'Adobe InDesign 15.1 (Macintosh)',
  'creationdate': '2025-07-10T16:11:16+09:00',
  'total_pages': 87,
  'producer': 'Adobe PDF Library 15.0',
  'trapped': '/False',
  'page_label': '1',
  'page': 0,
  'moddate': '2025-09-04T16:51:11+09:00',
  'source': '../../data/Samsung_Electronics_Sustainability_Report_2025_KOR.pdf'},
 {'source': '../../data/Samsung_Electronics_Sustainability_Report_2025_KOR.pdf',
  'creationdate': '2025-07-10T16:11:16+09:00',
  'producer': 'Adobe PDF Library 15.0',
  'moddate': '2025-09-04T16:51:11+09:00',
  'trapped': '/False',
  'total_pages': 87,
  'creator': 'Adobe InDesign 15.1 (Macintosh)',
  'page_label': '2',
  'page': 1},
 {'creator': 'Adobe InDesign 15.1 (Macintosh)',
  'source': '../../data/Samsung_Electronics_Sustainability_Report_2025_KOR.pdf',
  'page_label': '3',
  'page': 2,
  'producer': 'Adobe PDF Library 15.0',
  'moddate': '2025-09-04T16:51:11+09:00',
  'total_pages': 87,
  'trapped': '/False',
  'creationdate': '2025-07-10T16:11:1

In [19]:
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

In [20]:
# vectorstroe를 처음에 만들고 더이상 vectorstore에 넣는 과정을 반복하지 않기위해
# 미리만든 chunk를 쓰지않고 chunk와 같은 Document list를 만듦
from langchain_core.documents import Document

bm_doc = []
for content, meta in zip(db_docs["documents"], db_docs["metadatas"]):
    bm_doc.append(Document(page_content=content, metadata=meta))
bm_doc[:3]

[Document(metadata={'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2025-07-10T16:11:16+09:00', 'total_pages': 87, 'producer': 'Adobe PDF Library 15.0', 'trapped': '/False', 'page_label': '1', 'page': 0, 'moddate': '2025-09-04T16:51:11+09:00', 'source': '../../data/Samsung_Electronics_Sustainability_Report_2025_KOR.pdf'}, page_content='삼성전자 지속가능경영보고서 2025\nA Journey  Towards \n a Sustainable Future\nA Journey  Towards\n a Sustainable Future'),
 Document(metadata={'source': '../../data/Samsung_Electronics_Sustainability_Report_2025_KOR.pdf', 'creationdate': '2025-07-10T16:11:16+09:00', 'producer': 'Adobe PDF Library 15.0', 'moddate': '2025-09-04T16:51:11+09:00', 'trapped': '/False', 'total_pages': 87, 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'page_label': '2', 'page': 1}, page_content='삼성전자 지속가능경영보고서 2025 02AppendixFacts & Figures PrinciplePlanet PeopleOur Company삼성전자 지속가능경영보고서 2025 02\nA Journey  Towards \n a Sustainable Future\nA Journey  Towards \n a Sustainable F

In [33]:
bm25 = BM25Retriever.from_documents(
    bm_doc
) # 키워드 기반 검색기
bm25.k = 5

In [28]:
# 벡터 검색기 + bm25 = 하이브리드 검색기 완성
ret_hybrid = EnsembleRetriever(
    retrievers= [ret_similarity, bm25],
    weights=[0.7, 0.3]
)

In [38]:
question = "삼성 전자의 2025년 전망은?"
result = ret_hybrid.invoke(question)
result

[Document(id='c014ad36-114e-461b-a287-facd432d0733', metadata={'trapped': '/False', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'producer': 'Adobe PDF Library 15.0', 'moddate': '2025-09-04T16:51:11+09:00', 'page_label': '1', 'total_pages': 87, 'creationdate': '2025-07-10T16:11:16+09:00', 'source': '../../data/Samsung_Electronics_Sustainability_Report_2025_KOR.pdf', 'page': 0}, page_content='삼성전자 지속가능경영보고서 2025\nA Journey  Towards \n a Sustainable Future\nA Journey  Towards\n a Sustainable Future'),
 Document(id='9bc4d95a-6e29-4002-84a5-1c25ba54baee', metadata={'moddate': '2025-09-04T16:51:11+09:00', 'creationdate': '2025-07-10T16:11:16+09:00', 'page': 3, 'trapped': '/False', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'source': '../../data/Samsung_Electronics_Sustainability_Report_2025_KOR.pdf', 'page_label': '4', 'total_pages': 87, 'producer': 'Adobe PDF Library 15.0'}, page_content="삼성전자 지속가능경영보고서 2025\n04\nOur Company AppendixFacts & Figures PrinciplePlanet People\n주주, 고객, 협력회사,

In [39]:
rich_docs(result, title="하이브리드 기반 top10개 확인")

                                           하이브리드 기반 top10개 확인                                            
┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  # ┃ Source                                           ┃ Page ┃ Preview                                          ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1 │ Samsung_Electronics_Sustainability_Report_2025_… │    1 │ 삼성전자 지속가능경영보고서 2025 A Journey       │
│    │                                                  │      │ Towards   a Sustainable Future A Journey         │
│    │                                                  │      │ Towards  a Sustainable Future                    │
│  2 │ Samsung_Electronics_Sustainability_Report_2025_… │    4 │ 삼성전자 지속가능경영보고서 2025 04 Our Company  │
│    │                                                  │      │ AppendixFacts & Figures PrinciplePlanet People   │
│    │                                                  │      │ 주주, 고객, 협력회사, 그리고 임직원 여러분,      │
│    │                                                  │      │ 2024년은 글로벌 지정학적 리스크와 AI 기술의 성 … │
│    │                                                  │      │ …                                                │
│  3 │ Samsung_Electronics_Sustainability_Report_2025_… │   76 │ 삼성전자 지속가능경영보고서 2025 76 독립된       │
│    │                                                  │      │ 인증인의 인증보고서 Our Company AppendixFacts &  │
│    │                                                  │      │ Figures PrinciplePlanet People                   │
│  4 │ Samsung_Electronics_Sustainability_Report_2025_… │   77 │ 삼성전자 지속가능경영보고서 2025 77 Scope 1, 2   │
│    │                                                  │      │ 온실가스 배출량 검증 의견서 Our Company          │
│    │                                                  │      │ AppendixFacts & Figures PrinciplePlanet People   │
│  5 │ Samsung_Electronics_Sustainability_Report_2025_… │   86 │ 삼성전자 지속가능경영보고서 2025 86              │
│    │                                                  │      │ 삼성전자주식회사는 경제·사회·환경적 가치 창출    │
│    │                                                  │      │ 성과를 다양한 이해관계자와 투명하게 소통하기     │
│    │                                                  │      │ 위해 2025년 열여덟 번째 지속가능경영보고서를     │
│    │                                                  │      │ 발간합니다. 작성 기준 본 보고서는 지속가능경영   │
│    │                                                  │      │ 보고 기준인 GRI(G…                               │
│  6 │ Samsung_Electronics_Sustainability_Report_2025_… │   52 │ 삼성전자 지속가능경영보고서 2025 52 삼성         │
│    │                                                  │      │ 드림클래스 삼성 드림클래스는 임직원의 후원과     │
│    │                                                  │      │ 참여를 바탕으로 교육 여건이 부족한  국내         │
│    │                                                  │      │ 중학생들이 꿈을 찾고 미래를 준비할 수 있도록     │
│    │                                                  │      │ 멘토링과 맞춤형  진로 교육을 제공하는 프         │
│    │                                                  │      │ 로그램으로 대학생 멘 토, …                       │
│  7 │ Samsung_Electronics_Sustainability_Report_2025_… │   63 │ 삼성전자 지속가능경영보고서 2025 63 사회성과     │
│    │                                                  │      │ 준법·윤리경영 2022년 2023년 2024년 [컴플라이언 … │
│    │                                                  │      │ 교육] 컴플라이언스 교육 1) 명 126,867  138,742   │
│    │                                                  │      │ 138,414 [부정 예방 교육] 부정 예방 교육 2) 명    │
│    │                                                  │      │ 254,045  …                                       │
│  8 │ Samsung_Electronics_Sustainability_Report_2025_… │   51 │ 프로그램을 운영하고 있습니다. 추진 방향          │
│    │                                                  │      │ 삼성전자는 ‘함께가요 미래로! Enabling P eople’   │
│    │                                                  │      │ 비전 아래 , 교육의  기회에서 소외되는 학생 없이  │
│    │   

## 2.압축 검색기 (Compression retriever)
- 검색된 문서가 길 때 -> llm을 이용해서 내용을 압축해보기
- 문서 내용이 너무 파편화 되어있는 경우 -> 압축 진행 -> 찌꺼기를 제거

In [35]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor, LLMChainFilter
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model_name="gpt-4.1-mini",
                   temperature=0)

In [43]:
# 압축기 생성
compressor = LLMChainExtractor.from_llm(model)

# 압축 검색기 생성 = 유사도 검색 --> 문서 내용 압축
compress_ret = ContextualCompressionRetriever(
    base_retriever = ret_mmr,
    base_compressor=compressor,
)

In [44]:
question = "삼성 전자의 목표와 기준 년도만 간단히 알려줘"
compress_result = compress_ret.invoke(question)
compress_result

[Document(metadata={'source': '../../data/Samsung_Electronics_Sustainability_Report_2025_KOR.pdf', 'moddate': '2025-09-04T16:51:11+09:00', 'creationdate': '2025-07-10T16:11:16+09:00', 'trapped': '/False', 'page': 3, 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'total_pages': 87, 'producer': 'Adobe PDF Library 15.0', 'page_label': '4'}, page_content="삼성전자는 2022년 9월 발표한 '新환경경영전략'을 기반으로 탄소중립 달성, 자원순환 극대화, 그리고 기술 혁신을 통한 환경 난제 해결을 위해 노력하고 있습니다.  \nDX(Device eXperience)부문은 2030년 탄소중립 달성을 목표로  \nDS(Device Solutions)부문은 2050년 탄소중립 달성을 목표로"),
 Document(metadata={'source': '../../data/Samsung_Electronics_Sustainability_Report_2025_KOR.pdf', 'total_pages': 87, 'moddate': '2025-09-04T16:51:11+09:00', 'page_label': '1', 'trapped': '/False', 'creationdate': '2025-07-10T16:11:16+09:00', 'producer': 'Adobe PDF Library 15.0', 'page': 0, 'creator': 'Adobe InDesign 15.1 (Macintosh)'}, page_content='삼성전자 지속가능경영보고서 2025'),
 Document(metadata={'moddate': '2025-09-04T16:51:11+09:00', 'creator': 'Adobe InDes

In [45]:
rich_docs(compress_result, title="압축된 문서")

                                                    압축된 문서                                                    
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ Source                                           ┃ Page ┃ Preview                                           ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ Samsung_Electronics_Sustainability_Report_2025_… │    4 │ 삼성전자는 2022년 9월 발표한 '新환경경영전략'을   │
│   │                                                  │      │ 기반으로 탄소중립 달성, 자원순환 극대화, 그리고   │
│   │                                                  │      │ 기술 혁신을 통한 환경 난제 해결을 위해 노력하고   │
│   │                                                  │      │ 있습니다.   DX(Device eXperience)부문은 2030년    │
│   │                                                  │      │ 탄소중립 달성을 목표로   …                        │
│ 2 │ Samsung_Electronics_Sustainability_Report_2025_… │    1 │ 삼성전자 지속가능경영보고서 2025                  │
│ 3 │ Samsung_Electronics_Sustainability_Report_2025_… │   40 │ 삼성전자는 노동조합(한국)과의 교섭을 통해 2025년  │
│   │                                                  │      │ 3월 임금·단체협약을 체결하였으며, 협약 내용에     │
│   │                                                  │      │ 따라 근로조건을 개선하고 노동조합 활동 인프라를   │
│   │                                                  │      │ 확대하는 등 협력적 노사관계를 구축하기 위해       │
│   │                                                  │      │ 노력하고 있습니다.                                │
│ 4 │ Samsung_Electronics_Sustainability_Report_2025_… │   53 │ 삼성전자는 글로벌 개인정보보호 정책을 수립하고    │
│   │                                                  │      │ 국가에 따라 다른 주요  법과 제도를 반영해 지역별  │
│   │                                                  │      │ 상황에 적합한 정책을 운영하며, 임직원들의         │
│   │                                                  │      │ 개인정보보호 실천을 강화하기 위하여 '개인정보보 … │
│   │                                                  │      │ 임직원 가이드라인' 과 '개인정보처리 위탁 가이드'  │
│   │                                                  │      │ 등 관…                                            │
└───┴──────────────────────────────────────────────────┴──────┴───────────────────────────────────────────────────┘

### 2-2 임베딩 기반 경량 압축(비용 X)

In [51]:
from langchain.retrievers.document_compressors import EmbeddingsFilter
ret_mmr = load_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k":20, "lambda_mult": 0.5}
)

emb_filter = EmbeddingsFilter(
    embeddings=embedding,
    similarity_threshold=0.2,
)

comp_embed = ContextualCompressionRetriever(
    base_retriever=ret_mmr,
    base_compressor=emb_filter
)

comp_embed_result = comp_embed.invoke(question)

In [52]:
rich_docs(comp_embed_result, title="필터 기반 검색")

                                                  필터 기반 검색                                                   
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ Source                                           ┃ Page ┃ Preview                                           ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ Samsung_Electronics_Sustainability_Report_2025_… │    4 │ 삼성전자 지속가능경영보고서 2025 04 Our Company   │
│   │                                                  │      │ AppendixFacts & Figures PrinciplePlanet People    │
│   │                                                  │      │ 주주, 고객, 협력회사, 그리고 임직원 여러분,       │
│   │                                                  │      │ 2024년은 글로벌 지정학적 리스크와 AI 기술의 성장  │
│   │                                                  │      │ …                                                 │
│ 2 │ Samsung_Electronics_Sustainability_Report_2025_… │    1 │ 삼성전자 지속가능경영보고서 2025 A Journey        │
│   │                                                  │      │ Towards   a Sustainable Future A Journey  Towards │
│   │                                                  │      │ a Sustainable Future                              │
│ 3 │ Samsung_Electronics_Sustainability_Report_2025_… │   40 │ 삼성전자 지속가능경영보고서 2025 40 활동 결사의   │
│   │                                                  │      │ 자유와 단체교섭 보장 결사의 자 유는 근로자가      │
│   │                                                  │      │ 노동조합을 결성하 거나 가 입할 수 있는            │
│   │                                                  │      │ 권리입니다. 단체교섭은 사용자와 노동조합이 건     │
│   │                                                  │      │ 설적인 논의를 통해  근로조건을 확립하고 근로자의  │
│   │                                                  │      │ 기회균등을 …                                      │
│ 4 │ Samsung_Electronics_Sustainability_Report_2025_… │   53 │ 삼성전자 지속가능경영보고서 2025 53 추진 방향     │
│   │                                                  │      │ 삼성전자는 개인정보를 최소한의 범위 내에서        │
│   │                                                  │      │ 투명하게 수집하고, 안전하게 처리하며, 사용자의    │
│   │                                                  │      │ 선택을 최우선으로 존중합니다. 또한 한발 앞서      │
│   │                                                  │      │ 잠재적인  위험 요소를 사전에 식별하고 이에 맞는   │
│   │                                                  │      │ 첨단 보안 …                                       │
│ 5 │ Samsung_Electronics_Sustainability_Report_2025_… │   86 │ 비록 삼성전자주식회사는 지속가능경영보고서의 미 … │
│   │                                                  │      │ 예측 진술이 시의성 있고 합리적인 정보, 가정  및   │
│   │                                                  │      │ 믿음에 기반한다고 판단하지만, 이러한 미래 예측    │
│   │                                                  │      │ 진술(그리고 이를 이루는 정보, 가정 및 믿음)은     │
│   │                                                  │      │ 다양한 요인, 리스크, 불확실성의 영향권에          │
│   │                                                  │      │ 있으므로…                                         │
└───┴──────────────────────────────────────────────────┴──────┴───────────────────────────────────────────────────┘

## 3. 리랭커

In [ ]:
# uv add sentence-transformers langchain_huggingface

- 우선 검색기로 후보군을 추출 (like fetch_k) (10개~30개)

In [55]:
ret_similarity = load_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}
)

In [57]:
question="삼성의 현재 기업 규모가 어느정도야?"
ret_similarity_result = ret_similarity.invoke(question)
rich_docs(ret_similarity_result, title="삼성 기업규모 docs")

                                                삼성 기업규모 docs                                                 
┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  # ┃ Source                                           ┃ Page ┃ Preview                                          ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1 │ Samsung_Electronics_Sustainability_Report_2025_… │    5 │ 삼성전자 지속가능경영보고서 2025 05 Our Company  │
│    │                                                  │      │ AppendixFacts & Figures PrinciplePlanet People   │
│    │                                                  │      │ 회사소개 About Us 삼성전자주식회사(이하          │
│    │                                                  │      │ 삼성전자)는 인재와 기술을 기반으로 최고의 제품 … │
│    │                                                  │      │ 서비스를…                                        │
│  2 │ Samsung_Electronics_Sustainability_Report_2025_… │    4 │ 삼성전자 지속가능경영보고서 2025 04 Our Company  │
│    │                                                  │      │ AppendixFacts & Figures PrinciplePlanet People   │
│    │                                                  │      │ 주주, 고객, 협력회사, 그리고 임직원 여러분,      │
│    │                                                  │      │ 2024년은 글로벌 지정학적 리스크와 AI 기술의 성 … │
│    │                                                  │      │ …                                                │
│  3 │ Samsung_Electronics_Sustainability_Report_2025_… │   86 │ 삼성전자 지속가능경영보고서 2025 86              │
│    │                                                  │      │ 삼성전자주식회사는 경제·사회·환경적 가치 창출    │
│    │                                                  │      │ 성과를 다양한 이해관계자와 투명하게 소통하기     │
│    │                                                  │      │ 위해 2025년 열여덟 번째 지속가능경영보고서를     │
│    │                                                  │      │ 발간합니다. 작성 기준 본 보고서는 지속가능경영   │
│    │                                                  │      │ 보고 기준인 GRI(G…                               │
│  4 │ Samsung_Electronics_Sustainability_Report_2025_… │   50 │ 삼성전자 지속가능경영보고서 2025 50 안전보건     │
│    │                                                  │      │ 삼성전자는 DX부문의 Global EHS실장, DS부문의     │
│    │                                                  │      │ 글로벌 제조&인프라  총괄장인 CSO(Chief Safety    │
│    │                                                  │      │ Officer)를 중심으로 2030년까지 상주  협력회사    │
│    │                                                  │      │ 안전보건 역량과 관리체계 …                       │
│  5 │ Samsung_Electronics_Sustainability_Report_2025_… │   59 │ 삼성전자 지속가능경영보고서 2025 59 추진 체계    │
│    │                                                  │      │ 삼성전자는 기업의 책임 경영을 실현하기 위해      │
│    │                                                  │      │ 이사회와 주요 산하 위원회(경영위원회,            │
│    │                                                  │      │ 지속가능경영위원회, 감사위원회, 내부거래위원회   │
│    │                                                  │      │ 등)를 중심으로  준법과 윤리경영 체계를           │
│    │                                                  │      │ 관리·감독합니다. 이러한 거…                      │
│  6 │ Samsung_Electronics_Sustainability_Report_2025_… │   76 │ 삼성전자 지속가능경영보고서 2025 76 독립된       │
│    │                                                  │      │ 인증인의 인증보고서 Our Company AppendixFacts &  │
│    │                                                  │      │ Figures PrinciplePlanet People                   │
│  7 │ Samsung_Electronics_Sustainability_Report_2025_… │   86 │ 비록 삼성전자주식회사는 지속가능경영보고서의     │
│    │                                                  │      │ 미래 예측 진술이 시의성 있고 합리적인 정보, 가 … │
│    │                                                  │      │ 및 믿음에 기반한다고 판단하지만, 이러한 미래     │
│    │                                                  │      │ 예측 진술(그리고 이를 이루는 정보, 가정 및       │
│    │                                                  │   

In [58]:
from langchain_community.cross_encoders.huggingface import HuggingFaceCrossEncoder
from langchain.retrievers.document_compressors import CrossEncoderReranker

In [59]:
hf_cross_encoder = HuggingFaceCrossEncoder(
    model_name = "cross-encoder/ms-marco-MiniLM-L6-v2",
    model_kwargs = {
        "device": "cuda",
        "max_length": 512,
    }
)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

c:\Users\user\potenup\python7month\LangChainProject\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not instal

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [61]:
compressor = CrossEncoderReranker(
    model = hf_cross_encoder,
    top_n=10
)

reranker_retriever = ContextualCompressionRetriever(
    base_retriever = ret_similarity,
    base_compressor = compressor,
)
reranker_result = reranker_retriever.invoke(question)
rich_docs(reranker_result, title="리랭크 결과")

                                                    리랭크 결과                                                    
┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  # ┃ Source                                           ┃ Page ┃ Preview                                          ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1 │ Samsung_Electronics_Sustainability_Report_2025_… │   59 │ 삼성전자 지속가능경영보고서 2025 59 추진 체계    │
│    │                                                  │      │ 삼성전자는 기업의 책임 경영을 실현하기 위해      │
│    │                                                  │      │ 이사회와 주요 산하 위원회(경영위원회,            │
│    │                                                  │      │ 지속가능경영위원회, 감사위원회, 내부거래위원회   │
│    │                                                  │      │ 등)를 중심으로  준법과 윤리경영 체계를           │
│    │                                                  │      │ 관리·감독합니다. 이러한 거…                      │
│  2 │ Samsung_Electronics_Sustainability_Report_2025_… │   86 │ 비록 삼성전자주식회사는 지속가능경영보고서의     │
│    │                                                  │      │ 미래 예측 진술이 시의성 있고 합리적인 정보, 가 … │
│    │                                                  │      │ 및 믿음에 기반한다고 판단하지만, 이러한 미래     │
│    │                                                  │      │ 예측 진술(그리고 이를 이루는 정보, 가정 및       │
│    │                                                  │      │ 믿음)은  다양한 요인, 리스크, 불확실성의         │
│    │                                                  │      │ 영향권에 있으므로…                               │
│  3 │ Samsung_Electronics_Sustainability_Report_2025_… │   86 │ 삼성전자 지속가능경영보고서 2025 86              │
│    │                                                  │      │ 삼성전자주식회사는 경제·사회·환경적 가치 창출    │
│    │                                                  │      │ 성과를 다양한 이해관계자와 투명하게 소통하기     │
│    │                                                  │      │ 위해 2025년 열여덟 번째 지속가능경영보고서를     │
│    │                                                  │      │ 발간합니다. 작성 기준 본 보고서는 지속가능경영   │
│    │                                                  │      │ 보고 기준인 GRI(G…                               │
│  4 │ Samsung_Electronics_Sustainability_Report_2025_… │    4 │ 삼성전자 지속가능경영보고서 2025 04 Our Company  │
│    │                                                  │      │ AppendixFacts & Figures PrinciplePlanet People   │
│    │                                                  │      │ 주주, 고객, 협력회사, 그리고 임직원 여러분,      │
│    │                                                  │      │ 2024년은 글로벌 지정학적 리스크와 AI 기술의 성 … │
│    │                                                  │      │ …                                                │
│  5 │ Samsung_Electronics_Sustainability_Report_2025_… │   50 │ 삼성전자 지속가능경영보고서 2025 50 안전보건     │
│    │                                                  │      │ 삼성전자는 DX부문의 Global EHS실장, DS부문의     │
│    │                                                  │      │ 글로벌 제조&인프라  총괄장인 CSO(Chief Safety    │
│    │                                                  │      │ Officer)를 중심으로 2030년까지 상주  협력회사    │
│    │                                                  │      │ 안전보건 역량과 관리체계 …                       │
│  6 │ Samsung_Electronics_Sustainability_Report_2025_… │    5 │ 삼성전자 지속가능경영보고서 2025 05 Our Company  │
│    │                                                  │      │ AppendixFacts & Figures PrinciplePlanet People   │
│    │                                                  │      │ 회사소개 About Us 삼성전자주식회사(이하          │
│    │                                                  │      │ 삼성전자)는 인재와 기술을 기반으로 최고의 제품 … │
│    │                                                  │      │ 서비스를…                                        │
│  7 │ Samsung_Electronics_Sustainability_Report_2025_… │   85 │ 삼성전자 지속가능경영보고서 2025 85 코드 공시    │
│    │                                                  │      │ 항목 참고 페이지 

## 4. 리오더
- 리랭크와 같이 사용
- 리랭크에서는 맥락(내용의 흐름) 고려 x

In [62]:
from langchain_community.document_transformers import LongContextReorder

reorder = LongContextReorder()
reordered_result = reorder.transform_documents(reranker_result)

rich_docs(reordered_result, title="리오더 결과")

                                                    리오더 결과                                                    
┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  # ┃ Source                                           ┃ Page ┃ Preview                                          ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1 │ Samsung_Electronics_Sustainability_Report_2025_… │   86 │ 비록 삼성전자주식회사는 지속가능경영보고서의     │
│    │                                                  │      │ 미래 예측 진술이 시의성 있고 합리적인 정보, 가 … │
│    │                                                  │      │ 및 믿음에 기반한다고 판단하지만, 이러한 미래     │
│    │                                                  │      │ 예측 진술(그리고 이를 이루는 정보, 가정 및       │
│    │                                                  │      │ 믿음)은  다양한 요인, 리스크, 불확실성의         │
│    │                                                  │      │ 영향권에 있으므로…                               │
│  2 │ Samsung_Electronics_Sustainability_Report_2025_… │    4 │ 삼성전자 지속가능경영보고서 2025 04 Our Company  │
│    │                                                  │      │ AppendixFacts & Figures PrinciplePlanet People   │
│    │                                                  │      │ 주주, 고객, 협력회사, 그리고 임직원 여러분,      │
│    │                                                  │      │ 2024년은 글로벌 지정학적 리스크와 AI 기술의 성 … │
│    │                                                  │      │ …                                                │
│  3 │ Samsung_Electronics_Sustainability_Report_2025_… │    5 │ 삼성전자 지속가능경영보고서 2025 05 Our Company  │
│    │                                                  │      │ AppendixFacts & Figures PrinciplePlanet People   │
│    │                                                  │      │ 회사소개 About Us 삼성전자주식회사(이하          │
│    │                                                  │      │ 삼성전자)는 인재와 기술을 기반으로 최고의 제품 … │
│    │                                                  │      │ 서비스를…                                        │
│  4 │ Samsung_Electronics_Sustainability_Report_2025_… │   34 │ 삼성전자 지속가능경영보고서 2025 34 임직원       │
│    │                                                  │      │ 공급망 사회공헌 개인정보보호와 보안 제품 품질과  │
│    │                                                  │      │ 안전 35 45 51 53 55 People 사회적 책임을 다하며  │
│    │                                                  │      │ 미래로 함께 나아갑니다. Our Company              │
│    │                                                  │      │ AppendixFacts & Figures…                         │
│  5 │ Samsung_Electronics_Sustainability_Report_2025_… │   76 │ 삼성전자 지속가능경영보고서 2025 76 독립된       │
│    │                                                  │      │ 인증인의 인증보고서 Our Company AppendixFacts &  │
│    │                                                  │      │ Figures PrinciplePlanet People                   │
│  6 │ Samsung_Electronics_Sustainability_Report_2025_… │   86 │ ·  삼성전자주식회사 지속가능경영 웹사이트        │
│    │                                                  │      │ http://www.samsung.com/sec/sustainability/main · │
│    │                                                  │      │ 삼성전자주식회사 IR 웹사이트                     │
│    │                                                  │      │ http://www.samsung.com/sec/ir ·                  │
│    │                                                  │      │ 삼성전자주식회사 뉴…                             │
│  7 │ Samsung_Electronics_Sustainability_Report_2025_… │   85 │ 삼성전자 지속가능경영보고서 2025 85 코드 공시    │
│    │                                                  │      │ 항목 참고 페이지 및 답변 온실가스 배출           │
│    │                                                  │      │ TC-SC-110a.1 (1) Scope 1 총 배출량, (2) PFCs     │
│    │                                                  │      │ 배출량 P.68, P.72 TC-SC-110a.2 Scope 1 배출량    │
│    │                                                  │      │ 관리, 감축 …   